In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add larnd-sim to path
sys.path.insert(0, str(Path.cwd().parent))

from larndsim.far_field.signal_calculation import *
from larndsim.consts import detector
from larndsim.config import get_config

print("Imports successful!")

In [ ]:
def launch_far_field_dipole_signal_calculation(
    voxel_pos, pixel_pos, voxel_charge, pixel_categories, z_anode, z_cathode, v_drift, tick_size, n_ticks, 
    n_terms=5, C=None, bx=16, by=16
):
    """
    Launch CUDA kernel for far-field dipole induced current calculation with time ticks.
    Uses 2D grid/block launch for (pixel, tick) and sums over voxels in each thread.
    (USED FOR TESTING/VALIDATION ONLY)
    
    Args:
        voxel_pos: (n_voxels, 3) array (initial positions)
        pixel_pos: (n_pixels, 2) array
        voxel_charge: (n_voxels,) array (charge in each voxel, in units of electrons)
        pixel_categories: (n_pixels,) array with values 0=INDUCTION, 1=COLLECTION, 2=NEIGHBOR
        z_anode, z_cathode: float
        v_drift: float
        tick_size: float (us)
        n_ticks: int
        n_terms: int (default 5)
        C: float (default: use RESPONSE_SAMPLING from detector)
        bx, by: block dimensions for (pixels, ticks) (default 16x16)
        
    Returns:
        output: (n_pixels, n_ticks) CuPy array - total induced current per pixel per tick
    """
    n_pixels = pixel_pos.shape[0]
    output = cp.zeros((n_pixels, n_ticks), dtype=cp.float32)
    blockspergrid = (
        (n_pixels + bx - 1) // bx,
        (n_ticks + by - 1) // by
    )
    threadsperblock = (bx, by)

    prev_n_terms = ff_induction.DIPOLE_N_TERMS
    ff_induction.DIPOLE_N_TERMS = n_terms
    prev_v_drift = detector.V_DRIFT
    detector.V_DRIFT = v_drift
    prev_tick_size = detector.TIME_SAMPLING
    detector.TIME_SAMPLING = tick_size

    calculate_ff_voxels[blockspergrid, threadsperblock](
        cp.asarray(voxel_pos[:,0]),
        cp.asarray(voxel_pos[:,1]),
        cp.asarray(voxel_pos[:,2]),
        cp.asarray(voxel_charge, dtype=cp.float32),
        cp.asarray(pixel_pos[:,0]),
        cp.asarray(pixel_pos[:,1]),
        cp.asarray(pixel_categories, dtype=cp.int32),
        float(z_anode), float(z_cathode), output
    )

    ff_induction.DIPOLE_N_TERMS = prev_n_terms
    detector.V_DRIFT = prev_v_drift
    detector.TIME_SAMPLING = prev_tick_size

    return output

In [ ]:
# CPU Reference Implementation
def dipole_term_z(x, y, z):
    r2 = x*x + y*y + z*z
    if r2 < 1e-20:
        return 0.0
    r = np.sqrt(r2)
    # Correct dipole field derivative: (r² - 3z²)/r⁵
    return (r2 - 3.0 * z * z) / (r2 * r2 * r)

def farfield_dWdz_series(x, y, z, drift_length, n_terms):
    accum = dipole_term_z(x, y, z)
    for n in range(1, n_terms + 1):
        z_plus = z + 2.0 * n * drift_length
        z_minus = z - 2.0 * n * drift_length
        accum += dipole_term_z(x, y, z_plus)
        accum += dipole_term_z(x, y, z_minus)
    return accum

def far_field_current(x, y, z, v_d, drift_length, n_terms, norm_C):
    # Negative sign accounts for electron charge (q < 0)
    return -v_d * norm_C * farfield_dWdz_series(x, y, z, drift_length, n_terms)

# Vectorize for array operations
far_field_current_vec = np.vectorize(far_field_current)

print("CPU reference functions defined!")

In [ ]:
# Load detector configuration
response_file = np.load('/global/homes/j/jchakran/myscratch/dev2/larnd-sim-example/larnd-sim/larndsim/bin/response_37_v2d_fsd_ndlar_full.npz', allow_pickle=True)
response_file_old = np.load('/global/homes/j/jchakran/myscratch/dev2/larnd-sim-example/larnd-sim/larndsim/bin/response_38_v2b_full.npz', allow_pickle=True)

DT = float(response_file['time_tick'])
BIN_SIZE = float(response_file['bin_size'])
DRIFT_LENGTH = float(response_file['drift_length'])
RESPONSE = response_file['response']
RESPONSE_OLD = response_file_old['response']
MAX_RESPONSE_TIME = RESPONSE.shape[2] * DT

DIFF_N_SIGMAS = 5

cfg = get_config('fsd')
from larndsim import consts
consts.load_properties(cfg['DET_PROPERTIES'], cfg['PIXEL_LAYOUT'], 
                      None, cfg['SIM_PROPERTIES'])
from larndsim.consts import detector, sim

# Physics parameters
V_DRIFT = detector.V_DRIFT  # cm/µs
PIXEL_PITCH = detector.PIXEL_PITCH  # cm
TPC_BORDERS = detector.TPC_BORDERS

# Use first TPC for testing
plane_id = 0
z_anode = float(TPC_BORDERS[plane_id][2][0])
z_cathode = float(TPC_BORDERS[plane_id][2][1])
DRIFT_LENGTH = abs(z_cathode - z_anode)

# Simulation parameters
N_DIPOLE_TERMS = 5
NORM_C = 0.03
N_TICKS = RESPONSE.shape[2]

DRIFT_TIME = DRIFT_LENGTH / V_DRIFT  # us

print(f"Drift length: {DRIFT_LENGTH:.2f} cm")
print(f"Drift velocity: {V_DRIFT:.4f} cm/µs")
print(f"Anode at z = {z_anode:.2f} cm")
print(f"Cathode at z = {z_cathode:.2f} cm")
print(f"Pixel pitch: {PIXEL_PITCH:.2f} cm")
print(f"Tick size: {DT:.2f} us")

In [ ]:
# Define test positions
# Voxel at (0, 0, cathode) - electron starts at cathode, drifts toward anode
x, y = 0.55, 0.55
voxel_pos = np.array([[x, y, z_cathode]], dtype=np.float32)
voxel_charge = np.array([1.], dtype=np.float32) # unit charge

# Pixel at origin
pixel_pos = np.array([[0.0, 0.0]], dtype=np.float32)

# CPU calculation
print("Computing CPU reference...")
i_far_field_cpu = []
for it in range(N_TICKS):
    t = it * DT
    # Electron drifts from cathode toward anode
    if z_cathode > z_anode:
        z = z_cathode - V_DRIFT * t
    else:
        z = z_cathode + V_DRIFT * t
    # Position relative to pixel (at origin)
    z_rel = z - z_anode
    i_val = far_field_current(x, y, z_rel, V_DRIFT, DRIFT_LENGTH, N_DIPOLE_TERMS*10, NORM_C)
    i_far_field_cpu.append(i_val)
i_far_field_cpu = np.array(i_far_field_cpu)

# GPU calculation
print("Computing GPU...")
output_gpu = launch_far_field_dipole_signal_calculation(
    voxel_pos, pixel_pos, voxel_charge, z_anode, z_cathode, V_DRIFT, DT, N_TICKS, 
    n_terms=N_DIPOLE_TERMS, C=NORM_C
)
i_far_field_gpu = cp.asnumpy(output_gpu[0])  

# Compare
plt.figure(figsize=(12, 6))
time_axis = np.arange(N_TICKS) * DT
plt.plot(time_axis, i_far_field_cpu, 'b:', label='CPU Reference', linewidth=2)
plt.plot(time_axis, i_far_field_gpu, 'r--', label='GPU Implementation', linewidth=1.5)
print(int(x/BIN_SIZE), int(y/BIN_SIZE))
plt.plot(time_axis, RESPONSE[int(x/BIN_SIZE), int(y/BIN_SIZE), :], label='Response file')


plt.xlabel('Time (µs)', fontsize=12)
plt.ylabel('Induced Current', fontsize=12)
plt.title('Far-Field Dipole Signal: CPU vs GPU', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
